In [ ]:
# === Config (edit here) ===
ROOT = "/data1/tpz/nwm-main/results/nwm_cdit_airvln_16/airvln_16/CEM_N10_K10_RS1_rep3_OPT1/editor/run_006"            # 根目录：包含多个子目录，每个子目录有 metadata.json 与 frames/*.png
MASTER_OUT = "master.mp4"  # 输出视频文件名
FPS = 2                    # 合成视频帧率
PAUSE_SEC = 2.0   

In [ ]:
# === Imports ===
import os, json
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw

import matplotlib.pyplot as plt
from matplotlib import gridspec, animation
from IPython.display import HTML

In [ ]:
from PIL import Image, ImageDraw
import numpy as np
from pathlib import Path
import json
import matplotlib.pyplot as plt
from matplotlib import gridspec, animation

def build_master_animation(root: str, fps: int = 2, pause_sec: float = 0.0, overlay_text: bool = True):
    """
    Build the master animation from all runs under `root`.

    Changes vs. previous:
      - GT is FIXED: prefer `run_dir/frames/goal.png` per run;
        fallback to `ROOT/frames/goal.png`; otherwise use gray placeholder.
      - Pause between trajectories is supported (PAUSE overlay on left panel).
      - UI: Left=Current View, Middle=3D overlay (black=done, red=current), Right=Target (GT for loss).

    Returns: (fig, anim)
    """
    root = Path(root)
    run_dirs = sorted({p.parent for p in root.rglob("metadata.json")})
    assert run_dirs, f"No metadata.json found under {root}"

    # Optional global GT fallback
    global_gt_path = root / "frames" / "goal.png"
    global_gt_img = None
    if global_gt_path.exists():
        global_gt_img = Image.open(global_gt_path).convert("RGB")

    runs = []
    max_h = max_w = 0
    for rd in run_dirs:
        meta = json.load(open(rd / "metadata.json", "r", encoding="utf-8"))

        frames_rgb, poses, targets_rgb = [], [], []
        for f in meta["frames"]:
            # frame image
            rel = f["png"]
            p1 = rd / rel              # run/frames/xxx.png
            p2 = rd / Path(rel).name   # run/xxx.png
            png = p1 if p1.exists() else p2
            assert png.exists(), f"Frame not found: {png}"

            img = Image.open(png).convert("RGB")
            if overlay_text:
                step = f.get("frame", 0); act = f.get("action", "")
                draw = ImageDraw.Draw(img)
                draw.text((10, 10), f"Step {step}: {act}", fill=(255, 0, 0))
            frames_rgb.append(np.array(img))

            # pose
            p = f["pose"]
            poses.append((p["x"], p["y"], p["z"], p["theta_rad"]))

            # temporary placeholder for GT; we’ll finalize size after we know (h, w)
            targets_rgb.append(None)

        # Normalize sizes within the run (use first frame’s size as reference)
        h, w = frames_rgb[0].shape[:2]
        for i in range(len(frames_rgb)):
            if frames_rgb[i].shape[:2] != (h, w):
                frames_rgb[i] = np.array(Image.fromarray(frames_rgb[i]).resize((w, h)))

        # Finalize GT for this run:
        gt_resized = np.array(global_gt_img.resize((w, h)))
        targets_rgb = [gt_resized for _ in targets_rgb]  # same GT for all frames in this run

        max_h, max_w = max(max_h, h), max(max_w, w)
        runs.append((frames_rgb, poses, targets_rgb))

    # Global axis limits for the 3D plot
    all_xyz = np.array([p[:3] for _, poses, _ in runs for p in poses], dtype=np.float32)
    minv = all_xyz.min(axis=0); maxv = all_xyz.max(axis=0)
    pad = 0.05 * max(maxv - minv)
    if pad == 0: pad = 1.0

    # Build schedule with pause frames
    pause_frames = int(round(max(0.0, pause_sec) * max(1, fps)))
    schedule = []  # (run_idx, frame_idx, is_pause, tick)
    for i, (frames_rgb, _, _) in enumerate(runs):
        T = len(frames_rgb)
        for fidx in range(T):
            schedule.append((i, fidx, False, 0))
        schedule.append((i, T - 1, False, 0))  # one freeze frame
        for k in range(1, pause_frames + 1):
            schedule.append((i, T - 1, True, k))

    # Figure with three subplots: [Left image | Middle 3D | Right GT]
    fig = plt.figure(figsize=(15, 6))
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1.5, 1])

    ax_img = fig.add_subplot(gs[0])
    ax_img.axis('off')
    ax_img.set_title("Current View")
    im = ax_img.imshow(runs[0][0][0])

    ax3d = fig.add_subplot(gs[1], projection='3d')
    ax3d.set_title("Overlaid Trajectories (black=done, red=current)")
    ax3d.set_xlim(minv[0] - pad, maxv[0] + pad)
    ax3d.set_ylim(minv[1] - pad, maxv[1] + pad)
    ax3d.set_zlim(minv[2] - pad, maxv[2] + pad)

    ax_gt = fig.add_subplot(gs[2])
    ax_gt.axis('off')
    ax_gt.set_title("Target (GT for loss)")
    im_gt = ax_gt.imshow(runs[0][2][0])

    done_lines = []
    current_line = None
    prev_run = None

    # pause overlay on the left image
    pause_text = ax_img.text(
        0.5, 0.1, "", color="yellow", fontsize=16, weight="bold",
        ha="center", va="center", transform=ax_img.transAxes,
        bbox=dict(boxstyle="round,pad=0.3", fc="black", alpha=0.5)
    )

    def init():
        nonlocal current_line, prev_run
        im.set_data(runs[0][0][0])
        im_gt.set_data(runs[0][2][0])
        current_line, = ax3d.plot([], [], [], '-', lw=3, color='red')
        pause_text.set_text("")
        prev_run = 0
        return [im, im_gt, current_line, pause_text]

    def animate(k):
        nonlocal current_line, prev_run
        r, f, is_pause, tick = schedule[k]
        frames_rgb, poses, targets_rgb = runs[r]

        # on run switch: finalize previous (red) to black and start a new red
        if prev_run is not None and prev_run != r:
            prev_frames_rgb, prev_poses, prev_targets_rgb = runs[prev_run]
            xs, ys, zs = zip(*[p[:3] for p in prev_poses])
            done_lines.append(ax3d.plot(xs, ys, zs, '-', lw=2, color='black', alpha=0.8)[0])
            current_line.remove()
            current_line, = ax3d.plot([], [], [], '-', lw=3, color='red')
            prev_run = r

        # images
        im.set_data(frames_rgb[f])
        im_gt.set_data(targets_rgb[f])

        # current partial path (red)
        xs, ys, zs = zip(*[p[:3] for p in poses[:f+1]])
        current_line.set_data(xs, ys)
        current_line.set_3d_properties(zs)

        # pause overlay
        if is_pause and pause_frames > 0:
            secs_left = (pause_frames - tick + 1) / max(1, fps)
            pause_text.set_text(f"PAUSE {secs_left:.1f}s")
        else:
            pause_text.set_text("")

        return [im, im_gt, current_line, pause_text] + done_lines

    anim = animation.FuncAnimation(
        fig, animate, init_func=init,
        frames=len(schedule),
        interval=max(1, 1000 // max(1, fps)),
        blit=False
    )
    return fig, anim

In [ ]:
# === Preview in notebook (interactive) ===
fig, anim = build_master_animation(ROOT, fps=FPS, pause_sec=PAUSE_SEC, overlay_text=True)
HTML(anim.to_jshtml())

In [ ]:
anim.save(MASTER_OUT, writer='ffmpeg', fps=FPS)
plt.close(fig)
print(f"[OK] Master saved: {MASTER_OUT}")